<a href="https://colab.research.google.com/github/Dharmikreddy16/MLA0204/blob/main/decision_tree_based_ID3_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import math

# Define a class for the decision tree node
class DecisionTreeNode:
    def __init__(self, attribute=None, label=None, branches=None):
      self.attribute = attribute  # the attribute used to split the data
      self.label = label  # the label assigned to this node
      self.branches = branches if branches is not None else {}  # the branches of the decision tree

# Define a function to calculate the entropy of a dataset
def entropy(data):
    target = data['Target']
    n = len(target)
    unique, counts = np.unique(target, return_counts=True)
    entropy = 0
    for i in range(len(unique)):
        p = counts[i] / n
        entropy -= p * math.log2(p)
    return entropy

# Define a function to calculate the information gain of an attribute
def information_gain(data, attribute):
    n = len(data)
    values = data[attribute].unique()
    entropy_s = entropy(data)
    entropy_attr = 0
    for value in values:
        subset = data[data[attribute] == value]
        subset_n = len(subset)
        subset_entropy = entropy(subset)
        entropy_attr += subset_n / n * subset_entropy
    return entropy_s - entropy_attr

# Define the ID3 algorithm
def id3(data, attributes):
    target = data['Target']
    # If all the examples have the same target value, return a leaf node with that value
    if len(target.unique()) == 1:
        return DecisionTreeNode(label=target.iloc[0])
    # If there are no attributes left to split on, return a leaf node with the most common target value
    if len(attributes) == 0:
        return DecisionTreeNode(label=target.value_counts().idxmax())
    # Otherwise, select the attribute with the highest information gain
    gains = {attr: information_gain(data, attr) for attr in attributes}
    best_attribute = max(gains, key=gains.get)
    # Create a new decision tree node with the selected attribute
    node = DecisionTreeNode(attribute=best_attribute, branches={})
    # Split the data based on the selected attribute and recursively build the tree
    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value].drop(best_attribute, axis=1)
        if len(subset) == 0:
            node.branches[value] = DecisionTreeNode(label=target.value_counts().idxmax())
        else:
            new_attributes = attributes.copy()
            new_attributes.remove(best_attribute)
            node.branches[value] = id3(subset, new_attributes)
    return node

# Define a function to classify a new sample using the decision tree
def classify(sample, tree):
    if tree.label is not None:
        return tree.label
    attribute = tree.attribute
    value = sample[attribute]
    if value not in tree.branches:
        return tree.branches[max(tree.branches.keys(), key=int)]
    subtree = tree.branches[value]
    return classify(sample, subtree)

# Load the dataset
data = pd.read_csv('play_tennis.csv')
# Split the dataset into attributes and target variable
attributes = data.columns[:-1].tolist()
# Build the decision tree using ID3 algorithm
root = id3(data, attributes)

# Manually enter features to test the decision tree
print("Please enter the features for the day:")
outlook = input("Outlook (sunny / overcast / rain): ").strip().lower()
temp = input("Temperature (hot / mild / cool): ").strip().lower()
humidity = input("Humidity (high / normal): ").strip().lower()
wind = input("Wind (strong / weak): ").strip().lower()

# Create a Series from manual input
manual_sample = pd.Series({
    'Outlook': outlook,
    'Temperature': temp,
    'Humidity': humidity,
    'Wind': wind
})

# Predict using the decision tree
manual_prediction = classify(manual_sample, root)
print("\n--- Classification Result ---")
print(f"Sample features:\n{manual_sample.to_string()}")
print(f"\nPrediction (Will they play tennis?): {manual_prediction}")

Please enter the features for the day:
Outlook (sunny / overcast / rain): sunny
Temperature (hot / mild / cool): hot
Humidity (high / normal): normal
Wind (strong / weak): weak

--- Classification Result ---
Sample features:
Outlook         sunny
Temperature       hot
Humidity       normal
Wind             weak

Prediction (Will they play tennis?): yes
